# Chapter 2 lab — Who calculates the price of a draft?

Draft educator companion v1 · 8 September 2026 · 90 minutes.

Read [the chapter](https://www.profrod.ai/book/ch02-shop-tools) alongside this lab. **Prerequisites:** Chapter 1 request/response boundaries; Python dictionaries and integer arithmetic.

You will build one explicitly scoped decision function, challenge it with independently authored cases, and trace the same concern through the cumulative runtime. The manuscript is where you build the full components; this notebook is a focused companion, not a claim that importing a runtime teaches its construction.

**Before running:** write your prediction. Keep the worked solution below closed until you have attempted the function. Download/open this notebook in an existing Jupyter environment using the book’s Python 3.14 interpreter after completing repository setup in the book conventions. Unlike the two Chapter 1 notebooks, this lab requires the local checkout and its locked book dependencies. It does not install packages, launch a hosted notebook, or require model credentials.

**Without a notebook server:** read and edit the cells in your editor, then run `uv run --python 3.14 python book/always_on/educator/run_lesson_v1.py --chapter 2 --output /tmp/lucy-ch02-class.json` from the repository root. Use a new output filename on each retained run. The runner executes the saved notebook and records student results separately from the worked example.


## 1. Predict (10 minutes)

Vanilla has two tubs and a target of eight. A model asks for six, then True, then seven. Which requests are valid, and what is the cost at 250 pence per tub?

Write both the expected result and the evidence that could disprove your explanation.


In [ ]:
import copy
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 14):
    raise RuntimeError("Use the book Python 3.14 environment for Chapters 2–16.")
# Open the notebook inside your source checkout, or set this path explicitly.
start = Path(os.environ.get("SOVEREIGN_AGENT_REPO", Path.cwd())).resolve()
ROOT = next(
    (p for p in (start, *start.parents) if (p / "book/always_on/checkpoints/ch02.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Set SOVEREIGN_AGENT_REPO to the Sovereign Agent checkout.")
CHECKPOINT = ROOT / "book/always_on/checkpoints/ch02.py"
EXPECTED_CHECKPOINT_SHA256 = "d5fc4f480494911ff70df299e1464b82290142ca2d81193c880be6f6a71f7ae8"
if hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError("Checkpoint version differs from this lesson; use its matching release.")
print("Chapter 2 checkpoint bytes match this lesson. No model or channel has been called.")

## 2. Build your decision (25 minutes)

Implement decide(case). Inputs on_hand, target and unit_pence are trusted nonnegative integers. quantity is untrusted. Return a DRAFT dictionary with quantity and total_pence only when quantity is a positive exact int equal to max(0, target-on_hand). Otherwise return REFUSED. Never modify the input or perform a purchase.

`decide` is your code. `grade` and the fixtures are supplied test infrastructure. The examples below specify expected answers independently; do not generate those answers with your function. An unimplemented starter is reported as NOT_SUBMITTED, never as a pass.


In [ ]:
def grade(candidate, cases):
    results = []
    for index, (case, expected) in enumerate(cases, 1):
        supplied = copy.deepcopy(case)
        try:
            observed = candidate(supplied)
        except NotImplementedError:
            results.append({"case": index, "status": "NOT_SUBMITTED"})
            continue
        except Exception as error:
            results.append({"case": index, "status": "FAILED", "error_type": type(error).__name__})
            continue
        try:
            passed = json.dumps(observed, sort_keys=True, allow_nan=False) == json.dumps(
                expected, sort_keys=True, allow_nan=False
            ) and json.dumps(supplied, sort_keys=True, allow_nan=False) == json.dumps(
                case, sort_keys=True, allow_nan=False
            )
        except (TypeError, ValueError):
            passed = False
        results.append(
            {
                "case": index,
                "status": "PASS" if passed else "FAILED",
                "expected": expected,
                "observed": observed,
            }
        )
    return results


def decide(case):
    # Replace this body with your implementation of the contract above.
    raise NotImplementedError("Write your function before consulting the worked solution.")

In [ ]:
CASES = [
    (
        {"on_hand": 2, "target": 8, "unit_pence": 250, "quantity": 6},
        {"status": "DRAFT", "quantity": 6, "total_pence": 1500},
    ),
    ({"on_hand": 2, "target": 8, "unit_pence": 250, "quantity": True}, "REFUSED"),
    ({"on_hand": 2, "target": 8, "unit_pence": 250, "quantity": 7}, "REFUSED"),
    ({"on_hand": 12, "target": 6, "unit_pence": 300, "quantity": 0}, "REFUSED"),
    ({"on_hand": 2, "target": 8, "unit_pence": 250, "quantity": "6"}, "REFUSED"),
]
submission_results = grade(decide, CASES)
print(json.dumps(submission_results, indent=2))

## 3. Inspect and run the cumulative reference (20 minutes)

learner/ch02.py: build_tools and Dispatcher.invoke. Trace schema validation → allowed tool → deterministic handler → structured result.

Open the named code before running it. Point to where an input reaches a decision and where that decision changes an observable result. The next cell executes the supplied chapter checkpoint; it is reference evidence, not a substitute for your implementation. Local supplier and worker processes use temporary state and are cleaned up by the checkpoint. Chapter 11 also launches a bounded local MCP process. Chapter 15 does not install a system service.


In [ ]:
# This supplied cumulative program is separate from grading your function.
# It uses fixture models/channels. Some chapters start local child processes.
# No --live, --telegram or --containers switch is added.
reference_environment = {
    k: v for k, v in os.environ.items() if k in {"PATH", "SYSTEMROOT", "TMPDIR", "LANG", "LC_ALL"}
}
reference_environment["PYTHONPATH"] = str(ROOT / "src")
reference_run = subprocess.run(
    [sys.executable, str(CHECKPOINT)],
    cwd=ROOT,
    env=reference_environment,
    capture_output=True,
    text=True,
    timeout=180,
    check=True,
)
print(reference_run.stdout)
EXPECTED_OBSERVATIONS = ["invalid_arguments", '"total_pence": 1500']
assert all(text in reference_run.stdout for text in EXPECTED_OBSERVATIONS)
print("REFERENCE_CHECKPOINT_PASSED — this is not your submission grade.")

## 4. Transfer the rule (20 minutes)

Lime has zero tubs, target four, and a price of 200 pence. Add a case for four tubs and a case where quantity is 4.0. Then adapt the actual Chapter 2 shop fixture and price map; name the separate place that validates the SKU.

Add your new case to TRANSFER_CASES, with an independently calculated expected answer. A blank list means the transfer remains unsubmitted. Describe one limit of your function before comparing it with the runtime.


In [ ]:
TRANSFER_CASES = []  # Add (input, expected) pairs after writing your prediction.
transfer_results = grade(decide, TRANSFER_CASES)
print(json.dumps(transfer_results, indent=2) if transfer_results else "TRANSFER_NOT_SUBMITTED")

## 5. Worked solution — reveal after attempting the task

The following function is an answer key, not a replacement for your submission. Its case results are recorded separately. Your teacher grades your original function, explanation and transfer case.

Four costs 800 pence; 4.0 is refused by the strict type rule. Add Lime to both products and PRICES, then use build_tools with the expanded fixture. Product identity and duplicate-SKU checks belong to build_tools/argument validation, outside this arithmetic function.


In [ ]:
def worked_decide(case):
    needed = max(0, case["target"] - case["on_hand"])
    quantity = case["quantity"]
    if type(quantity) is not int or quantity <= 0 or quantity != needed:
        return "REFUSED"
    return {"status": "DRAFT", "quantity": quantity, "total_pence": quantity * case["unit_pence"]}


worked_results = grade(worked_decide, CASES)
assert all(row["status"] == "PASS" for row in worked_results)
print("WORKED_EXAMPLE_PASSED; submission_results remains separate.")

## 6. Break the tempting implementation (10 minutes)

Explain why the following shortcut violates at least one case. Predict which case catches it before running. Then name a different defect the current cases might miss.


In [ ]:
def tempting_shortcut(case):
    return {
        "status": "DRAFT",
        "quantity": case["quantity"],
        "total_pence": case["quantity"] * case["unit_pence"],
    }


shortcut_results = grade(tempting_shortcut, CASES)
assert any(row["status"] == "FAILED" for row in shortcut_results)
print(json.dumps(shortcut_results, indent=2))

## Exit ticket (5 minutes)

Submit your prediction, original decide function, case results, one transfer case, and the runtime path you traced. Explain: (1) which boundary Python enforced, (2) what evidence came from the supplied program, and (3) what remains unproved.

**Misconception to resolve:** Python treats bool as a subclass of int. isinstance(True, int) cannot enforce the quantity contract. A well-formed tool result is still a draft.

**Scope of this lab:** The exercise isolates quantity arithmetic. The chapter builds dispatch, schemas and authority checks; completing this function does not implement that whole dispatcher.

A successful reference run or worked example does not establish learner mastery. Instructor guidance and answers are in the matching versioned guide.
